# Data Retrieval Polymarket

| Field | Details |
|---|---|
| **Time window** | 1 Jul 2024 – 5 Nov 2024 |
| **Source** | Polymarket — Gamma API (`gamma-api.polymarket.com`) |
| **Method** | REST API — JSON response with daily win probabilities |
| **Content** | Daily market-implied probability of winning per candidate |
| **Saved to** | `Data/1_Bronze/polymarket/polymarket_win_probabilities.csv` · `Data/1_Bronze/polymarket/polymarket_volume.csv` |

### polymarket_win_probabilities.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (YYYY-MM-DD) |
| `Trump (%)` | float | Market-implied probability of Trump winning (%) |
| `Harris (%)` | float | Market-implied probability of Harris winning (%) |

<!-- toc -->
## Contents
- [Setup](#setup)
- [1. Win Probabilities](#1-win-probabilities)
  - [polymarket_win_probabilities.csv columns](#polymarket_win_probabilitiescsv-columns)
- [2. Trading Volume](#2-trading-volume)
  - [polymarket_volume.csv columns](#polymarket_volumecsv-columns)


## Setup

In [ ]:
import requests
import json
import pandas as pd
import os

## 1. Win Probabilities

### polymarket_win_probabilities.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (YYYY-MM-DD) |
| `Trump (%)` | float | Market-implied probability of Trump winning (%) |
| `Harris (%)` | float | Market-implied probability of Harris winning (%) |

In [4]:
# Step 1: Fetch event and extract token IDs
slug = "presidential-election-winner-2024"
event = requests.get(f"https://gamma-api.polymarket.com/events?slug={slug}").json()[0]

def get_yes_token(markets, name):
    market = next(m for m in markets if name in m["question"])
    return json.loads(market["clobTokenIds"])[0]  # index 0 = Yes token

trump_token  = get_yes_token(event["markets"], "Trump")
harris_token = get_yes_token(event["markets"], "Harris")


In [5]:
# Step 2: Fetch daily price history and combine
def get_price_history_full(token_id, label):
    params = {"market": token_id, "interval": "max", "fidelity": 1440}
    resp = requests.get("https://clob.polymarket.com/prices-history", params=params).json()
    df = pd.DataFrame(resp.get("history", []))
    df["datetime_utc"] = pd.to_datetime(df["t"], unit="s")
    df["probability"]  = df["p"].astype(float) * 100
    # 00:00 UTC day D = closing price of day D-1 (US time)
    df["date"] = (df["datetime_utc"] - pd.Timedelta(days=1)).dt.normalize()
    return df[["date", "probability"]].rename(columns={"probability": label})

trump_df  = get_price_history_full(trump_token,  "Trump (%)")
harris_df = get_price_history_full(harris_token, "Harris (%)")

combined_df = pd.merge(trump_df, harris_df, on="date", how="outer").sort_values("date").reset_index(drop=True)
combined_df[["Trump (%)", "Harris (%)"]] = combined_df[["Trump (%)", "Harris (%)"]].round(2)


In [ ]:
final = combined_df[
    (combined_df["date"] >= "2024-07-01") & # for lags later
    (combined_df["date"] <= "2024-11-05")
].reset_index(drop=True)

output_path = "../Data/1_Bronze/polymarket/polymarket_win_probabilities.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
final.to_csv(output_path, index=False)
print(f"Saved {len(final)} rows → {output_path}")
final.head()

## 2. Trading Volume

| Field | Details |
|---|---|
| **Time window** | 1 Jul 2024 – 5 Nov 2024 |
| **Source** | Polymarket — Data API (`data-api.polymarket.com/v1/builders/volume`) |
| **Method** | REST API — `timePeriod=ALL`, summed across all builders per day |
| **Content** | Total daily USDC trading volume across the entire Polymarket platform |
| **Saved to** | `Data/1_Bronze/polymarket/polymarket_volume.csv` |

### polymarket_volume.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (YYYY-MM-DD) |
| `polymarket_volume_usdc` | float | Total USDC traded across all builders that day |

In [ ]:
# Step 4: Fetch daily trading volume via Polymarket Data API
# Endpoint: GET https://data-api.polymarket.com/v1/builders/volume?timePeriod=ALL
# Returns daily volume broken down by builder (front-end/app).
# We sum across all builders per day to get total platform volume.

resp = requests.get(
    "https://data-api.polymarket.com/v1/builders/volume",
    params={"timePeriod": "ALL"},
)
resp.raise_for_status()
records = resp.json()

volume_df = pd.DataFrame(records)
volume_df["date"] = pd.to_datetime(volume_df["dt"]).dt.normalize()
volume_df["volume"] = volume_df["volume"].astype(float)

daily_volume = (
    volume_df.groupby("date")["volume"]
    .sum()
    .reset_index()
    .rename(columns={"volume": "polymarket_volume_usdc"})
)

# Filter to campaign window
daily_volume = daily_volume[
    (daily_volume["date"] >= "2024-07-01") &
    (daily_volume["date"] <= "2024-11-05")
].reset_index(drop=True)

print(f"Fetched {len(daily_volume)} days of volume data")
daily_volume.head()

In [ ]:
output_path = "../Data/1_Bronze/polymarket/polymarket_volume.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
daily_volume.to_csv(output_path, index=False)
print(f"Saved {len(daily_volume)} rows → {output_path}")
daily_volume.describe()